In [ ]:
from src.constants import DATA_PROCESSED, PROFILES_TABLE

profiles_table = PROFILES_TABLE
output_dir = str(DATA_PROCESSED)

In [ ]:
from src.utils import load_env

load_env()
import json
from src.db.client import to_dataframe
from src.models.similarity import embed_bio_summaries

In [ ]:
# ── Bio embeddings: derived per-player artifact, kept out of the DB ──
# Sourced from gold.player_profiles.summary; saved as an independent
# player_id -> bio_* frame consumed by the NN (02_tune_nn) for the
# canonical player and opponent sides of each match row.

print("Computing bio embeddings from player_profiles.summary...")

profiles = to_dataframe(f"SELECT player_id, summary FROM {profiles_table}")
bio_df = embed_bio_summaries(profiles)
bio_feat_cols = [c for c in bio_df.columns if c != "player_id"]
bio_df.to_parquet(f"{output_dir}/bio_embeddings.parquet", index=False)

with open(f"{output_dir}/bio_feature_cols.json", "w") as f:
    json.dump(bio_feat_cols, f)

print(f"Saved {len(bio_df)} player embeddings, dim {len(bio_feat_cols)}")